# 02 · Evaluación de los tres enfoques

Conjunto test: 320 reseñas etiquetadas a mano (estratificadas por estrellas). Se regenera con `python -m resenas.evaluar` (sin Ollama: usa la caché de embeddings y las respuestas grabadas del LLM).

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
from resenas import config
ev = json.loads((config.DIR_REPORTES / 'evaluacion.json').read_text(encoding='utf-8'))
filas = []
for k, v in ev['enfoques'].items():
    fila = {'enfoque': k, **{m: v[m] for m in ['f1_aspectos','precision_aspectos','exhaustividad_aspectos','f1_aspecto_polaridad','exactitud_polaridad','exactitud_global','f1_macro_global']}}
    fila['seg_por_resena'] = v['costo']['segundos_por_resena']
    filas.append(fila)
pd.DataFrame(filas).set_index('enfoque').round(3)

,f1_aspectos,precision_aspectos,exhaustividad_aspectos,f1_aspecto_polaridad,exactitud_polaridad,exactitud_global,f1_macro_global,seg_por_resena
enfoque,,,,,,,,
clasico,0.722,0.681,0.769,0.550,0.762,0.797,0.700,0.001
embeddings,0.739,0.698,0.784,0.621,0.841,0.816,0.750,0.966
llm,0.688,0.699,0.678,0.629,0.914,0.694,0.618,19.509


## Intervalos de confianza (bootstrap de reseñas, 1000 réplicas)

In [2]:
pd.DataFrame({k: {m: f'{lo:.3f} – {hi:.3f}' for m, (lo, hi) in v['ic95'].items()} for k, v in ev['enfoques'].items()})

,clasico,embeddings,llm
f1_aspectos,0.695 – 0.747,0.709 – 0.765,0.654 – 0.720
precision_aspectos,0.650 – 0.711,0.664 – 0.730,0.660 – 0.738
exhaustividad_aspectos,0.736 – 0.804,0.749 – 0.815,0.640 – 0.713
f1_aspecto_polaridad,0.513 – 0.585,0.584 – 0.654,0.589 – 0.664
exactitud_polaridad,0.721 – 0.802,0.805 – 0.877,0.884 – 0.941
exactitud_global,0.750 – 0.841,0.775 – 0.859,0.641 – 0.744


## F1 por aspecto

In [3]:
pd.DataFrame({k: {a: d['f1'] for a, d in v['por_aspecto'].items()} for k, v in ev['enfoques'].items()}).assign(soporte=pd.Series(ev['soporte_test'])).sort_values('soporte', ascending=False).round(2)

,clasico,embeddings,llm,soporte
ajuste,0.72,0.76,0.73,102
calidad,0.75,0.75,0.71,99
facilidad_uso,0.72,0.72,0.64,63
diseno,0.60,0.58,0.62,49
carga,0.74,0.77,0.76,46
precio,0.80,0.89,0.55,45
sonido,0.90,0.91,0.82,33
bateria,0.78,0.75,0.77,31
envio,0.70,0.77,0.56,23
pantalla,0.71,0.57,0.67,22


## Por estrellas
Las reseñas de 1–2★ tienen más aspectos y más largos; las de 5★ muchas veces solo elogian en general.

In [4]:
pd.DataFrame({(k, e): {m: d[m] for m in ['f1_aspectos','exactitud_polaridad','exactitud_global']} for k, v in ev['enfoques'].items() for e, d in v['por_estrellas'].items()}).T.unstack(0).round(2)

f1_aspectos                  exactitud_polaridad                  exactitud_global                 
      clasico embeddings   llm             clasico embeddings   llm          clasico embeddings   llm
1        0.70       0.68  0.71                0.80       0.88  0.97             0.82       0.81  0.96
2        0.71       0.77  0.58                0.73       0.84  0.91             0.54       0.72  0.83
3        0.73       0.79  0.73                0.56       0.70  0.82             0.72       0.67  0.72
4        0.74       0.76  0.70                0.73       0.85  0.86             0.73       0.78  0.39
5        0.73       0.73  0.69                0.89       0.89  0.97             0.98       0.96  0.60

## Costo

In [5]:
pd.DataFrame({k: v['costo'] for k, v in ev['enfoques'].items()}).T

,segundos_por_resena,umbral_clausula,C_aspecto,umbral,f1_cv_dev,segundos_clasificadores,textos_por_resena,segundos_por_texto,mediana_segundos,p90_segundos,tokens_salida_medios,errores,citas_no_literales,menciones,n
clasico,0.001405,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
embeddings,0.965867,NaN,8.0,0.6,0.587593,0.001782,4.9225,0.195853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
llm,19.509250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.965,32.954,108.259375,0.0,16.0,529.0,320.0


## Acuerdo entre las dos pasadas de etiquetado (160 reseñas del test)

In [6]:
a = ev['acuerdo_anotacion']
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in a.items() if k != 'kappa_por_aspecto'}

{'n_resenas': 160,
 'f1_aspectos_entre_pasadas': 0.963,
 'kappa_aspectos_medio': 0.96,
 'acuerdo_polaridad': 0.977,
 'kappa_polaridad': 0.955,
 'kappa_global': 0.969,
 'acuerdo_global': 0.981}

## Parte en español (80 reseñas; ningún enfoque se ajustó con español)

In [7]:
pd.DataFrame({k: {m: v[m] for m in ['f1_aspectos','f1_aspecto_polaridad','exactitud_polaridad','exactitud_global']} for k, v in ev['espanol']['enfoques'].items()}).T.round(3)

,f1_aspectos,f1_aspecto_polaridad,exactitud_polaridad,exactitud_global
clasico,0.500,0.419,0.837,0.925
embeddings,0.704,0.638,0.907,0.912
llm,0.689,0.660,0.958,0.575


## Errores típicos
Reseñas del test donde los enfoques no coinciden con la verdad.

In [8]:
from resenas.etiquetas import leer_jsonl, cargar_muestra
# Sin el texto de la reseña (no se versiona por la licencia de los datos): solo metadatos y etiquetas.
muestra = cargar_muestra()
preds = leer_jsonl(config.DIR_REPORTES / 'predicciones_test.jsonl')
def asp(d):
    return ', '.join(f"{m['aspecto']}{'+' if m['polaridad']=='positivo' else '-' if m['polaridad']=='negativo' else '0'}" for m in d['aspectos']) or '—'
filas = [{'id': p['id'], 'estrellas': muestra[p['id']]['estrellas'], 'tipo': muestra[p['id']]['tipo_producto'],
          'verdad': asp(p['verdad']), 'clásico': asp(p['clasico']), 'embeddings': asp(p['embeddings']), 'llm': asp(p['llm'])} for p in preds]
df = pd.DataFrame(filas)
df[df.verdad != df.llm].head(12)

,id,estrellas,tipo,verdad,clásico,embeddings,llm
1,R017565,4,Protectores de pantalla,"pantalla+, calidad-","proteccion+, pantalla+, diseno0","pantalla-, calidad-, ajuste0","pantalla+, facilidad_uso-, calidad-"
2,R052128,4,Accesorios para auto,"calidad-, atencion+","calidad-, atencion-, ajuste-",calidad-,"calidad-, ajuste-"
4,R034965,5,Cables y adaptadores,envio+,"envio+, atencion+",envio+,—
5,R050301,5,Accesorios de smartwatch,"facilidad_uso+, pantalla+","pantalla+, facilidad_uso+","ajuste+, diseno+, facilidad_uso+",facilidad_uso+
6,R048278,4,Accesorios de smartwatch,diseno+,—,—,—
7,R024268,4,Accesorios de smartwatch,"precio+, ajuste+, facilidad_uso+","calidad+, precio+, ajuste+, facilidad_uso+","calidad+, ajuste+, precio+, facilidad_uso+","facilidad_uso+, ajuste+, precio+"
8,R051427,5,Auriculares Bluetooth,"sonido+, diseno+, bateria+","bateria+, sonido+","bateria+, sonido+",bateria+
9,R046156,5,Accesorios de smartwatch,"diseno+, facilidad_uso+, ajuste+","facilidad_uso+, diseno0, envio+, ajuste+","ajuste+, diseno+, facilidad_uso+","diseno+, ajuste+"
11,R017922,1,Auriculares Bluetooth,"facilidad_uso-, precio-","facilidad_uso-, precio-","precio-, facilidad_uso-","carga-, precio-"
13,R052721,1,Accesorios de smartwatch,"pantalla-, precio-","pantalla-, precio-",precio-,"pantalla-, calidad-"


In [9]:
det = pd.read_csv(config.DIR_REPORTES / 'evaluacion_detalle.csv')
# ¿qué aspectos inventa cada enfoque? (falsos positivos por aspecto)
pd.DataFrame({k: det[det.verdad.isna() & det[k].notna()].aspecto.value_counts() for k in ['clasico','embeddings','llm']}).fillna(0).astype(int)

,clasico,embeddings,llm
aspecto,,,
ajuste,13,25,20
atencion,9,3,17
bateria,5,4,6
calidad,22,40,39
carga,32,16,17
diseno,21,29,9
envio,12,6,0
facilidad_uso,19,25,14
pantalla,9,8,6
